# Hafta 13 — Zaman Serileri ve LSTM: Şebeke Yükü Tahmini

Veri: `sebeke_yuku.csv` (2025, saatlik yük MW + sıcaklık + takvim). Hedef: 1 saat ve 24 saat ileri yük tahmini; naif → ridge → orman → LSTM; artıktan anomali.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F, time
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(0); np.random.seed(0); cihaz = "cuda" if torch.cuda.is_available() else "cpu"; print("cihaz:", cihaz)
# Colab: sebeke_yuku.csv'yi sol paneldeki Dosyalar'a yükleyin (veya from google.colab import files; files.upload())
df = pd.read_csv("sebeke_yuku.csv"); df["tarih"] = pd.to_datetime(df.tarih); y = df.yuk_MW.values; n = len(y); NTR = 24*300
print(df.shape, "eğitim saat:", NTR, "test saat:", n - NTR); df.head()

## 1. Veriye bakalım: yıl, iki hafta, günlük profil

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 7))
axs[0].plot(df.tarih, y, lw=.4); axs[0].set_ylabel("MW"); axs[0].set_title("Bir yıl")
s = slice(24*40, 24*54); axs[1].plot(df.tarih[s], y[s]); axs[1].set_title("İki hafta (40–54. gün)")
prof = df.groupby(["haftanin_gunu", "saat"]).yuk_MW.mean().unstack()
for d, lab in ((0, "Pzt"), (3, "Per"), (5, "Cmt"), (6, "Paz")): axs[2].plot(prof.columns, prof.loc[d], "-o", ms=3, label=lab)
axs[2].legend(); axs[2].set_xlabel("saat"); axs[2].set_title("Günlük profil"); plt.tight_layout(); plt.show()
print("hafta içi / Cmt / Paz ortalama:", [round(df.yuk_MW[df.haftanin_gunu == d].mean()) for d in (2, 5, 6)], " tatil ort:", round(df.yuk_MW[df.tatil == 1].mean()))

## 2. Otokorelasyon ve sıcaklık ilişkisi

In [ ]:
yc = y - y.mean(); acf = np.array([np.sum(yc[:n-k]*yc[k:])/np.sum(yc*yc) for k in range(200)])
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2))
axs[0].bar(range(200), acf, width=1); axs[0].set_xlabel("gecikme (saat)"); axs[0].set_title("ACF")
axs[1].scatter(df.sicaklik_C[::6], y[::6], s=4, alpha=.3); axs[1].set_xlabel("°C"); axs[1].set_ylabel("MW"); axs[1].set_title("yük–sıcaklık"); plt.show()
print("ACF(1) =", round(acf[1], 3), " ACF(24) =", round(acf[24], 3), " ACF(168) =", round(acf[168], 3), " ACF(12) =", round(acf[12], 3))

**Soru:** ACF(12) neden düşük hatta negatif? Hangi gecikmeleri özellik yapmalıyız?

## 3. Zaman bölmesi ve naif tabanlar (Örnek 13.1)

In [ ]:
def mape(a, b): return 100*np.mean(np.abs(a - b)/np.abs(a))
def mae(a, b): return np.mean(np.abs(a - b))
def ozellik(df, lags=(1, 2, 3, 24, 48, 168), ort=1):
    X = pd.DataFrame({f"lag{l}": df.yuk_MW.shift(l) for l in lags})
    X["sin_h"] = np.sin(2*np.pi*df.saat/24); X["cos_h"] = np.cos(2*np.pi*df.saat/24)
    X["hafta_sonu"] = (df.haftanin_gunu >= 5).astype(int); X["tatil"] = df.tatil; X["sicaklik"] = df.sicaklik_C
    X["ort24"] = df.yuk_MW.shift(ort).rolling(24).mean()          # DİKKAT: shift(ort) — hedefi içermesin
    return X
idx = np.arange(n)

In [ ]:
y_te = y[NTR:]
tabanlar = {"naif 1 s": y[NTR-1:n-1], "naif 24 s": y[NTR-24:n-24], "naif 168 s": y[NTR-168:n-168]}
for ad, t in tabanlar.items(): print(f"{ad:10s} MAE {mae(y_te, t):6.1f} MW   RMSE {np.sqrt(np.mean((y_te-t)**2)):6.1f}   MAPE {mape(y_te, t):.2f} %")
# Örnek 13.1
yy = np.array([700, 720, 760, 790, 780.]); nf = np.array([680, 700, 720, 760, 790.]); md_ = np.array([705, 715, 750, 800, 770.])
for ad, t in (("naif", nf), ("model", md_)): e = yy - t; print(f"{ad:6s} MAE {np.abs(e).mean():.1f}  RMSE {np.sqrt((e**2).mean()):.1f}  MAPE {100*np.mean(np.abs(e)/yy):.2f} %")

## 4. Gecikme özellikleri + ridge / rastgele orman (1 saat ileri)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
X = ozellik(df); ok = X.notna().all(axis=1).values; tr = ok & (idx < NTR); ts = ok & (idx >= NTR)
rd = Ridge(1.0).fit(X[tr], y[tr]); pr_rd = rd.predict(X[ts]); print("ridge  MAPE %.2f %%" % mape(y[ts], pr_rd))
rf = RandomForestRegressor(200, n_jobs=-1, random_state=0).fit(X[tr], y[tr]); pr_rf = rf.predict(X[ts]); print("orman  MAPE %.2f %%" % mape(y[ts], pr_rf))
print(pd.Series(rd.coef_, X.columns).round(2).to_string())

**Soru:** Ridge katsayılarında lag1 ve lag24'ün işareti/büyüklüğü ne anlatıyor? `ort24`'ü `df.yuk_MW.rolling(24).mean()` olarak (shift'siz) yazsaydınız MAPE ne olurdu ve bu neden 'hile'dir? Deneyin.

## 5. Ufuk: 24 saat ileri klasik model

In [ ]:
X24 = ozellik(df, lags=(24, 48, 168), ort=24); ok24 = X24.notna().all(axis=1).values; tr24 = ok24 & (idx < NTR); ts24 = ok24 & (idx >= NTR)
rd24 = Ridge(1.0).fit(X24[tr24], y[tr24]); print("ridge 24 s ileri MAPE %.2f %%   (naif 24 s: %.2f %%)" % (mape(y[ts24], rd24.predict(X24[ts24])), mape(y_te, tabanlar["naif 24 s"])))

## 6. Pencereleme (Örnek 13.2)

Özellikler: ölçekli yük, sin/cos saat, hafta sonu, ölçekli sıcaklık → F = 5. Ölçekleme yalnızca eğitim istatistikleriyle.

In [ ]:
mu, sd = y[:NTR].mean(), y[:NTR].std(); ys = (y - mu)/sd
feat = np.stack([ys, np.sin(2*np.pi*df.saat/24), np.cos(2*np.pi*df.saat/24), (df.haftanin_gunu >= 5).astype(float), (df.sicaklik_C - 15)/10], 1).astype(np.float32)
def pencere(a, hedef, i0, i1, W, h=1):
    Xs, Ys = [], []
    for i in range(i0 + W, i1 - h + 1): Xs.append(a[i-W:i]); Ys.append(hedef[i+h-1])
    return torch.tensor(np.array(Xs)), torch.tensor(np.array(Ys), dtype=torch.float32)
W = 168; Xtr_t, ytr_t = pencere(feat, ys, 0, NTR, W); Xte_t, yte_t = pencere(feat, ys, NTR - W, n, W)
print("eğitim:", Xtr_t.shape, " test:", Xte_t.shape, " N = L−W−h+1 =", NTR - W - 1 + 1)

## 7. RNN ileri geçişi elle (Örnek 13.3) ve parametre sayıları (Örnek 13.4)

In [ ]:
Wr, Ur, h = 0.5, 0.8, 0.0
for x in [1, 0, -1]: h = np.tanh(Wr*x + Ur*h); print(round(h, 3), end="  ")
print("\nŷ =", round(2*h, 3), " 0.8^10 =", round(0.8**10, 3))
for ad, m in (("RNN", nn.RNN(5, 32)), ("GRU", nn.GRU(5, 32)), ("LSTM", nn.LSTM(5, 32))): print(ad, sum(p.numel() for p in m.parameters()))
print("elle LSTM (tek sapma):", 4*(32*(5+32) + 32))

## 8. LSTM ile 1 saat ileri tahmin

In [ ]:
class LSTMNet(nn.Module):
    def __init__(self, nf, h=32, cikti=1):
        super().__init__(); self.lstm = nn.LSTM(nf, h, batch_first=True); self.out = nn.Linear(h, cikti)
    def forward(self, x):
        o, _ = self.lstm(x); return self.out(o[:, -1]).squeeze(-1)
def egit_dizi(model, Xtr, ytr, Xte, yte_gercek, epoch=10, lr=3e-3, bs=128, geri=lambda p: p*sd + mu):
    model = model.to(cihaz); opt = torch.optim.Adam(model.parameters(), lr); dl = DataLoader(TensorDataset(Xtr, ytr), bs, shuffle=True); t0 = time.time(); hist = []
    for ep in range(epoch):
        model.train()
        for xb, yb in dl: xb, yb = xb.to(cihaz), yb.to(cihaz); opt.zero_grad(); F.mse_loss(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad(): pr = geri(model(Xte.to(cihaz)).cpu().numpy())
        hist.append(mape(yte_gercek, pr)); print(f"epoch {ep+1:2d}  test MAPE {hist[-1]:.2f} %  ({time.time()-t0:.0f} s)")
    return pr, hist
torch.manual_seed(0); lstm = LSTMNet(5); pr_lstm, hist = egit_dizi(lstm, Xtr_t, ytr_t, Xte_t, y_te)

## 9. Karşılaştırma

In [ ]:
sonuc = {"naif 24 s": mape(y_te, tabanlar["naif 24 s"]), "naif 168 s": mape(y_te, tabanlar["naif 168 s"]), "ridge": mape(y[ts], pr_rd), "orman": mape(y[ts], pr_rf), "LSTM": mape(y_te, pr_lstm)}
print({k: round(v, 2) for k, v in sonuc.items()})
s = slice(24*20, 24*27); tt = df.tarih[NTR:][s]
plt.figure(figsize=(11, 3.5)); plt.plot(tt, y_te[s], "k", lw=1.6, label="gerçek"); plt.plot(tt, tabanlar["naif 168 s"][s], "--", color="gray", label="naif 168 s")
plt.plot(tt, pr_rd[s], label="ridge"); plt.plot(tt, pr_lstm[s], label="LSTM"); plt.legend(ncol=4); plt.ylabel("MW"); plt.title("Test döneminden bir hafta"); plt.show()

**Soru:** LSTM'yi 168 yerine W = 24 ile eğitin. MAPE nasıl değişir; neden? (Ne kadar geçmişe ihtiyacı var?)

## 10. 24 saat ileri: 24 çıktılı LSTM (doğrudan strateji)

In [ ]:
def pencere_cok(a, hedef, i0, i1, W, H=24):
    Xs, Ys = [], []
    for i in range(i0 + W, i1 - H + 1): Xs.append(a[i-W:i]); Ys.append(hedef[i:i+H])
    return torch.tensor(np.array(Xs)), torch.tensor(np.array(Ys), dtype=torch.float32)
Xtr24, ytr24 = pencere_cok(feat, ys, 0, NTR, W); Xte24, yte24 = pencere_cok(feat, ys, NTR - W, n, W)
gercek24 = yte24.numpy()*sd + mu
torch.manual_seed(0); lstm24 = LSTMNet(5, 64, cikti=24)
pr24, _ = egit_dizi(lstm24, Xtr24, ytr24, Xte24, gercek24, epoch=20, lr=1e-3)
print("24 çıktılı LSTM: tüm ufuk ort. MAPE %.2f %%  | yalnızca 24. saat: %.2f %%  | ridge 24 s: %.2f %%" % (mape(gercek24, pr24), mape(gercek24[:, -1], pr24[:, -1]), mape(y[ts24], rd24.predict(X24[ts24]))))
plt.plot(range(1, 25), [mape(gercek24[:, k], pr24[:, k]) for k in range(24)], "o-"); plt.xlabel("ufuk (saat)"); plt.ylabel("MAPE %"); plt.title("Ufka göre hata"); plt.grid(alpha=.3); plt.show()

## 11. Artıktan anomali tespiti

In [ ]:
art_tr = y[tr] - rd.predict(X[tr]); sigma = art_tr.std()
yy = y[ts].copy(); yy[24*40:24*40+6] -= 120                      # yapay 6 saatlik kesinti
z = (yy - pr_rd)/sigma; alarm = np.where(np.abs(z) > 4)[0]
print("σ_eğitim = %.1f MW, alarm sayısı: %d, alarm saatleri (test indeksi): %s" % (sigma, len(alarm), alarm[:10]))
fig, axs = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True); tt = df.tarih[ts]
axs[0].plot(tt, yy, lw=.7, label="gözlem"); axs[0].plot(tt, pr_rd, lw=.7, alpha=.7, label="tahmin"); axs[0].legend(); axs[0].set_ylabel("MW")
axs[1].plot(tt, z, lw=.7, color="k"); axs[1].axhline(4, ls="--", color="r"); axs[1].axhline(-4, ls="--", color="r"); axs[1].set_ylabel("z"); plt.show()

**Soru:** Eşiği 3'e indirirseniz yanlış alarm sayısı ne olur? Kesinti −40 MW olsaydı yakalanır mıydı? Kesintiden sonraki saatte lag1 kesintili değeri görür — tahmin ne yapar?

## 12. Alıştırmalar

**Alıştırma 1 (CV).** `sklearn.model_selection.TimeSeriesSplit(n_splits=5)` ile ridge ve ormanın kat kat MAPE'sini raporlayın (eğitim bölümü üzerinde). Katlar arasında fark neden var?

**Alıştırma 2 (mimari).** GRU ve 1-B CNN (Conv1d(5,16,7,stride=2) → ReLU → Conv1d(16,16,7,stride=2) → ReLU → Flatten → Linear(608,64) → ReLU → Linear) tabanlı tahmincileri LSTM ile aynı veride karşılaştırın: parametre sayısı, epoch süresi, MAPE.

**Alıştırma 3 (özyinelemeli vs doğrudan).** 1 saat ileri LSTM'i kendi tahminini girdi yaparak 24 adım uygulayın (özyinelemeli); 24 çıktılı modelle (doğrudan) ufka göre MAPE eğrilerini aynı grafiğe çizin.

**Alıştırma 4 (özellik kaldırma).** LSTM'i yalnızca yük (F = 1) ile eğitin; sıcaklık ve takvim olmadan MAPE ne olur? Hangi özellik en değerli?

**Alıştırma 5 (bonus, PV).** 5. haftanın `pv_uretim.csv`'sinde gündüz saatleri için 1 saat ileri üretim tahmini (ridge + gecikmeler). MAPE yerine MAE ya da nMAE (kurulu güce bölünmüş) kullanın ve nedenini yazın.